# MiniGrid RL

## Setup

In [ ]:
from pathlib import Path
import gymnasium as gym
from feature_extractor import MinigridFeaturesExtractor
from levels import LevelOne, LevelTwo, ProceduralLevel
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
import pygame
from minigrid.wrappers import ImgObsWrapper
from tqdm import tqdm
import torch
from minigrid.wrappers import ImgObsWrapper, RGBImgPartialObsWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv


## Level 1

In [ ]:
n_envs = 6
n_timesteps = 100_000
device = "cpu"
policy_kwargs = {
    "features_extractor_class": MinigridFeaturesExtractor,
    "features_extractor_kwargs": {"features_dim": 256},
}


def make_env():
    env = ProceduralLevel(render_mode="rgb_array", difficulty=200)
    env = ImgObsWrapper(env)
    return env

env = make_vec_env(
    make_env, 
    n_envs = n_envs, 
    vec_env_cls = SubprocVecEnv,
)

model_level_path = Path("models/proc_testmodel.zip")

if model_level_path.exists():
    model = PPO.load(model_level_path, env=env)
    print("Model loaded successfully. Continuing training...")
    model.learn(total_timesteps = (n_timesteps * n_envs), progress_bar = True, reset_num_timesteps=False)
    model.save(model_level_path)
else:
    print("No saved model found. Training a new model ...")
    model = PPO(
        "CnnPolicy", 
        env, 
        policy_kwargs = policy_kwargs, 
        verbose = 1, 
        device = device, 
    )
    model.learn(total_timesteps = (n_timesteps * n_envs), progress_bar = True)
    model.save(model_level_path)

### Train

### Verify

In [ ]:
env = ProceduralLevel(render_mode="human",difficulty=200)
env = ImgObsWrapper(env)
obs, info = env.reset()
while True:
    action, _ = model.predict(obs)
    obs, rewards, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break

In [ ]:
env.close()